# 02 — Exploratory Data Analysis & Visualization

Explore patterns in the cleaned customer-level dataset covering:
- Revenue trends over time
- Top products and countries
- Invoice value and purchase frequency distributions
- Correlation structure of numerical features


In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
os.makedirs('outputs/figures', exist_ok=True)
print(f'Working dir: {os.getcwd()}')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme(style='whitegrid', palette='husl')
SAVE = lambda name: plt.savefig(f'outputs/figures/{name}', dpi=150, bbox_inches='tight')

df = pd.read_csv('data/cleaned_retail_customers.csv', parse_dates=['InvoiceDate'])
print(f"Loaded {len(df):,} rows, {df['CustomerID'].nunique():,} customers")

## Descriptive Statistics

In [ ]:
print("Numerical summary:")
df[['Quantity','Price','TotalPrice']].describe().round(2)

In [ ]:
print("Categorical counts:")
for col in ['Country','Year']:
    print(f"\n{col}: {df[col].nunique()} unique")
    print(df[col].value_counts().head(5))

## Monthly Revenue Trend

In [ ]:
monthly = (df.groupby(df['InvoiceDate'].dt.to_period('M'))['TotalPrice']
           .sum().reset_index())
monthly['InvoiceDate'] = monthly['InvoiceDate'].astype(str)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly['InvoiceDate'], monthly['TotalPrice']/1e6,
        marker='o', linewidth=2, color='steelblue', markersize=5)
ax.fill_between(range(len(monthly)), monthly['TotalPrice']/1e6, alpha=0.15, color='steelblue')
ax.set_title('Monthly Revenue Trend (2009–2011)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£ Millions)')
plt.xticks(range(len(monthly)), monthly['InvoiceDate'], rotation=45, ha='right', fontsize=8)
plt.tight_layout()
SAVE('eda_monthly_revenue.png')
plt.show()

## Top 10 Products by Revenue

In [ ]:
top_products = df.groupby('Description')['TotalPrice'].sum().nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('Blues_r', 10)
bars = ax.barh(range(10), top_products.values/1e3, color=colors, edgecolor='white')
ax.set_yticks(range(10))
ax.set_yticklabels([t[:45] for t in top_products.index], fontsize=9)
ax.set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Revenue (£ Thousands)')
for bar, val in zip(bars, top_products.values/1e3):
    ax.text(val+0.3, bar.get_y()+bar.get_height()/2, f'£{val:.1f}k', va='center', fontsize=8)
plt.tight_layout()
SAVE('eda_top_products.png')
plt.show()

## Sales by Country (Top 10)

In [ ]:
top_countries = df.groupby('Country')['TotalPrice'].sum().nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('Set2', 10)
ax.barh(range(10), top_countries.values/1e6, color=colors, edgecolor='white')
ax.set_yticks(range(10))
ax.set_yticklabels(top_countries.index)
ax.set_title('Top 10 Countries by Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Revenue (£ Millions)')
plt.tight_layout()
SAVE('eda_countries.png')
plt.show()

## Invoice Value Distribution

In [ ]:
inv_totals = df.groupby('Invoice')['TotalPrice'].sum()
inv_totals = inv_totals[(inv_totals > 0) & (inv_totals < inv_totals.quantile(0.99))]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(inv_totals, bins=70, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(inv_totals.median(), color='red', linestyle='--', linewidth=1.8,
           label=f'Median: £{inv_totals.median():.0f}')
ax.axvline(inv_totals.mean(), color='orange', linestyle='--', linewidth=1.8,
           label=f'Mean: £{inv_totals.mean():.0f}')
ax.set_title('Invoice Value Distribution (excl. top 1%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Invoice Total (£)')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
SAVE('eda_invoice_dist.png')
plt.show()

## Customer Purchase Frequency

In [ ]:
cust_freq = df.groupby('CustomerID')['Invoice'].nunique()
cap = cust_freq.quantile(0.95)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(cust_freq.clip(upper=cap), bins=45, color='coral', edgecolor='white', alpha=0.85)
ax.set_title('Customer Purchase Frequency (clipped at 95th pct)', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Invoices per Customer')
ax.set_ylabel('Number of Customers')
ax.axvline(cust_freq.median(), color='navy', linestyle='--', linewidth=1.8,
           label=f'Median: {cust_freq.median():.0f} invoices')
ax.legend()
plt.tight_layout()
SAVE('eda_customer_freq.png')
plt.show()

## Correlation Heatmap

In [ ]:
num_cols = ['Quantity','Price','TotalPrice','Month','DayOfWeek','Hour']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap — Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
SAVE('eda_correlation.png')
plt.show()

## Revenue by Day of Week

In [ ]:
dow = df.groupby('DayOfWeek')['TotalPrice'].sum()
day_names = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

fig, ax = plt.subplots(figsize=(9, 5))
colors = sns.color_palette('Set3', 7)
bars = ax.bar([day_names[i] for i in dow.index], dow.values/1e6, color=colors, edgecolor='white', linewidth=1.2)
ax.set_title('Total Revenue by Day of Week', fontsize=14, fontweight='bold')
ax.set_ylabel('Revenue (£ Millions)')
for bar, val in zip(bars, dow.values/1e6):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
            f'£{val:.1f}M', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
SAVE('eda_dow_revenue.png')
plt.show()
print("All EDA figures saved to outputs/figures/")